# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainabaon/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zainabaon/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found — check path"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship
Starter data found. You're ready.


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm choosing Lane 2: Refresh / Content Opportunity Scoring. In notebooks 01 and 02 I already saw a hand-written rule score Precision@50 = 0.240 while a random forest scored 0.740 on the same task — a ~3x improvement in how many of the top 50 flagged pages were actually declining. That gap tells me there's real, learnable signal in this data beyond what a simple rule can capture, which is exactly what this lane is built to explore further. I also want to build on the depth-2 vs depth-4 decision tree comparison from notebook 02, where I saw firsthand that model complexity trades off against readability — a tension this lane forces you to navigate directly, since the output has to be an explainable ranked queue with reason codes, not a black box.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Question: Which content pages should be reviewed first for refresh, based on evidence they are visible (getting real exposure) but declining or underperforming?

Unit of analysis: one page (one row = one content item, at a point in time).

Output: a ranked queue of pages, ordered by a refresh-priority score, each with a reason code explaining why it was flagged (e.g. "stale_visible_page", "declining_with_demand").

Action: the content team reviews the top N pages (e.g. top 50, matching realistic review capacity) this week/sprint, instead of guessing which pages to refresh first.

Cost of a wrong call: a false positive wastes a reviewer's limited time on a page that wasn't actually a priority. A false negative means a genuinely declining, high-traffic page gets missed and keeps losing visibility unreviewed. Given limited review capacity, precision at the top of the queue matters more than overall accuracy.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I loaded the starter CSV and pulled three numbers that show why this lane has real signal worth building on.
54.2% of pages in the sample are currently declining (trend_direction == "down"), and 9,961 of those — 33.2% of the entire dataset — still get 500+ impressions in the last 90 days. This means declining pages aren't rare edge cases here; they're the single largest group in the data, and a third of all pages are both visible enough to matter and showing decline. Combined with the ~3x precision gap already observed between the hand-written rule and the random forest (0.240 vs 0.740), this confirms both that there's a large, real pool of candidates to rank and that a learned model can meaningfully outperform a simple rule at prioritizing them.

In [8]:
import os
print(os.getcwd())
print(os.listdir())

/content/flyrank-ml-internship/flyrank-ml-internship
['LICENSE', 'notebooks', 'skills', '.github', 'CLAUDE.md', 'data', '.git', 'submission', 'AGENTS.md', 'docs', 'requirements.txt', 'outputs', 'README.md', 'work', 'DATA_USE.md', 'GUIDE.md', '.gitignore', 'SETUP.md', 'scripts']


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 1: how much of the inventory is actually declining
print("Share of pages by trend_direction:")
print(df["trend_direction"].value_counts(normalize=True).round(3))

# Number 2: baseline vs model gap already observed in notebooks 01/02
print("\nFrom notebook 01: hand-written rule Precision@50 = 0.240")
print("Random forest Precision@50 = 0.740  (~3x improvement)")

# Number 3: how many declining pages are actually still visible (worth reviewing, not dead)
visible_declining = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 500)]
print(f"\nDeclining pages with >=500 impressions_90d: {len(visible_declining)} "
      f"({len(visible_declining)/len(df):.1%} of all pages)")

Share of pages by trend_direction:
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64

From notebook 01: hand-written rule Precision@50 = 0.240
Random forest Precision@50 = 0.740  (~3x improvement)

Declining pages with >=500 impressions_90d: 9961 (33.2% of all pages)


What I can claim: this dataset shows an observed, measurable gap between a hand-written rule and a learned model at ranking which pages are worth reviewing first (Precision@50: 0.240 vs 0.740). It also shows that declining, still-visible pages are a large and real group in this data — 33.2% of all pages — not a rare edge case. This is decision-support: it helps prioritize limited human review time, nothing more.

What I can't claim: I cannot claim that refreshing a flagged page will cause it to recover — proving that would require a causal experiment, not this observational data. I cannot claim to have discovered a Google ranking factor, or that this predicts how Google's algorithm works. The current label (trend_direction == "down") is a same-window proxy based on the current window, not a true future outcome. A stronger version of this lane would predict decline over a future window using only prior data, with a strict leakage audit, before making any prediction-style claims.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.